# Evaluating AI with `screamingface` — any benchmark, four steps (desired UX)

**Status: runnable UX contract.** Every cell executes top-to-bottom, no API keys.
Where the SDK surface doesn't exist yet, [`sf_stubs.py`](sf_stubs.py) patches the
*target* API onto today's SDK: real local machinery where we have it (the validated
rubric grader ships in this folder), and a `[stub · …]` log line linking to the
production implementation in
[screamingface-benchmarks](https://github.com/OpenMined/screamingface-benchmarks)
wherever it's pretending. As each SDK gap lands, its stub steps aside and the cells
keep working unchanged.

## Analogy: Eval = an exam, run by code
![The exam analogy over the four-seam pipeline](diagrams/sf-eval-pipeline.png)

This notebook shows the UX for a generalized benchmarking framework: **three completely
different benchmarks, the same four steps** — `load → run → grade → aggregate`.

| | worker (RUN) | examiner (GRADE) |
|---|---|---|
| **MedXpertQA** (sample rows) | 3-model panel, **majority vote** | answer key — pure code, free |
| **DRACO** (real 5-row slice) | 2-model panel, **a LLM synthesizer** merges drafts | weighted rubric — a pinned LLM judge, costs money |
| **HealthBench** (sample row) | same fusion | **same rubric judge, zero new code** — the genericity gate |

![One notebook, two benchmarks, same four steps](diagrams/sf-eval-notebook-flow.png)

## What's in this folder — mapped to the four steps

Everything here is a real artifact from our validated DRACO runs, and each folder is
one piece of the generalized pipeline:

| Folder | Pipeline step | What it is |
|---|---|---|
| `data/` | **LOAD** — the question paper | 5 real DRACO rows (`question` + rubric in `answer`). Exactly what `sf.benchmark()` loads. Private — the rubric is the answer key. |
| *(the SDK + url4 engine)* | **RUN** — the worker(s) | Not in this folder: the panel fan-out + synthesizer live in the `screamingface` package (installed by `uv sync`) and the url4 engine it talks to. |
| `benchmarking/` | **GRADE** — the examiner | The validated rubric grader ([`graders/rubric.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/rubric.py)), the judge prompts (official mode = DRACO paper verbatim), and the abort-vs-skip error classes. Exactly what `sf.RubricJudge` wraps. |
| `runners/` | **GRADE** — the money guard | [`spend_guard.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/runners/spend_guard.py), the grader's one dependency: fail-closed budget gating on every paid judge call (the $2,000-abort lesson). |
| `validated_run/` | **AGGREGATE** — the report card | The scores of record (`by_model.csv`: fusion 0.6862 vs best solo 0.5722) and the bill (`pipeline_metrics.json`: judge-dominated). What the compare cell reads. |
| `sf_stubs.py` | *(the shim)* | Patches the target API onto today's SDK so this notebook runs; every pretend step logs `[stub · …]`. |
| `diagrams/` | *(teaching aids)* | The figures embedded in this notebook. |

Setup once: `uv sync` in this folder, then pick the `.venv` kernel.

In [1]:
import screamingface as sf
import sf_stubs  # patches the target API onto today's SDK — see sf_stubs.py

[sf_stubs] screamingface 0.2.0 + target API patched: sf.benchmark, sf.Fusion (catalog fallback), fusion.evaluate(Benchmark), sf.RubricJudge, sf.MultipleChoice · extension seams: @sf.loader / @sf.runner / @sf.grader / @sf.aggregator


## 1 · LOAD — the question paper *(stubbed: SDK gap 1)*

One loader for every benchmark. A row is always the same shape:
`id · source · task_type · question · answer · metadata`. The `task_type` tells the
pipeline how the worker should work (`mcq`, `abstractive`, later `multi_turn`,
`agentic`, …) — an open string, so types that don't exist yet load without a schema
change.

⚠️ **The sealed envelope rule.** The `answer` field is the answer key — for DRACO it
IS the weighted rubric. It travels to the **examiner only**; the panel and synthesizer
see nothing but `question`. A worker who sees the rubric is cheating (DRACO panel
models with web search were literally *finding the rubric online* — hence the domain
blocklist in the config of record, `benchmarks_config/draco.yaml` in
[screamingface-benchmarks](https://github.com/OpenMined/screamingface-benchmarks) —
and don't post the JSONL anywhere crawlable).

In [2]:
bench_draco = sf.benchmark("data/draco-demo-slice-5.jsonl")   # real 5-row DRACO slice
bench_mx    = sf.benchmark("medxpertqa-sample")               # MCQ shape (synthetic rows)
bench_hb    = sf.benchmark("healthbench-sample")              # 2nd rubric benchmark (synthetic row)

print()
for b in (bench_draco, bench_mx, bench_hb):
    print(repr(b))

print("\nDRACO row 0 rubric — this is what 'graded' means:")
bench_draco[0].rubric.preview()

[stub · load] data/draco-demo-slice-5.jsonl → 5 rows · prod ingestion: https://github.com/OpenMined/screamingface-benchmarks/blob/main/data_ingestion/run_ingestion.py
[stub · load] 'medxpertqa-sample' → 2 synthetic sample rows (shape-faithful) · real loader: https://github.com/OpenMined/screamingface-benchmarks/blob/main/data_ingestion/hf_loader.py · real grader: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/mcq_exact.py
[stub · load] 'healthbench-sample' → 1 synthetic sample row (shape-faithful) · real grader: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/healthbench_rubric.py · real prompts: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/prompts/healthbench.py

Benchmark · draco (5 rows) · abstractive · graded by weighted rubric (LLM judge)
Benchmark · medxpertqa-sample (2 rows) · mcq · graded by answer key (pure code)
Benchmark · healthbench-sample (1 rows) · abstractive · gra

## 2 · COMPOSE the workers — a fusion is one url4 recipe *(real SDK)*

ScreamingFace's move on the worker side is deliberately simple: **ask several AIs
instead of one**, and merge. Two merge strategies, matched to the task type:

- `sf.MajorityVote` — countable answers (MCQ letters). Free, pure code.
- `sf.Synthesize(model)` — free-form reports have no vote to take; a synthesizer LLM
  merges the labeled drafts. One extra model call, part of the url4 graph.

Both compile to a **shareable url4 recipe** — the whole setup in one string. A
leaderboard row, a tweet, and a Colab cell are the same string; displaying it never
executes it.

In [3]:
# MCQ path — real sf.Fusion, catalog models, compiles to real url4
fusion_mcq = sf.Fusion(
    "frontier-trio",
    models=["codex/gpt-5.5", "gemini-cli/gemini-2.5-pro", "anthropic/claude-sonnet-4-6"],
    reducer=sf.MajorityVote(tie_breaker="codex/gpt-5.5"),
)
print("real url4:", fusion_mcq.url4[:120], "…\n")

# DRACO path — the validated lineup (0.6862 on the full 100 rows).
# These model ids aren't in the SDK catalog yet (gap 6) → preview-only recipe.
fusion_draco = sf.Fusion(
    "fable_plus_gpt",
    models=["anthropic/claude-fable-5", "openai/gpt-5.5"],
    reducer=sf.Synthesize("anthropic/claude-opus-4.8"),
)
print("\npreview recipe:", fusion_draco.url4[:160], "…")

real url4: (panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question', panel_2=/gemini/2.5($question)!'Answer the mul …

[stub · compose] unknown model: anthropic/claude-fable-5 → SDK gap 6 (catalog + engine routes); building a preview-only recipe instead

preview recipe: (panel_1=/anthropic/claude-fable-5($question)!'Answer the question', panel_2=/openai/gpt-5.5($question)!'Answer the question', fusion_answer=/anthropic/claude-o …


## 3 · RUN — the workers attempt every row *(stubbed: SDK gaps 2–3)*

`evaluate()` takes the **Benchmark object** (today's SDK only accepts the string
`"gpqa"`). Per row: panel fans out in parallel through the url4 engine, the reducer
merges, and — new — **every answer is kept** on `run.rows` so the examiner can grade
them afterwards. Here the engine call is a deterministic stand-in; the `[stub · run]`
lines show exactly what a real run would send.

The runner is typed to `row.question` only — the rubric physically cannot reach the
panel. Sealed envelope, enforced by construction.

*Honesty note (DRACO):* with web tools off, scores land well below the validated
chart — the 0.69 needed OpenRouter server-side `web_search`/`web_fetch` and
`max_tokens ≥ 8192`. Expected; label it.

In [4]:
run_mx = fusion_mcq.evaluate(bench_mx, seed=0)
print(repr(run_mx), "\n")

run_draco = fusion_draco.evaluate(bench_draco, seed=0)
print(repr(run_draco), "\n")

run_hb = fusion_draco.evaluate(bench_hb, seed=0)   # same fusion, different benchmark
print(repr(run_hb))

[stub · run] 2 rows × (3 panel + 0 synth) → 6 engine calls in a real run · every answer kept on run.rows
[stub · run] recipe: (panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question', panel_2=/gemini/2.5($question)!'Answ…
Run · frontier-trio · medxpertqa-sample · 2 rows · answer_calls=6 

[stub · run] 5 rows × (2 panel + 1 synth) → 15 engine calls in a real run · every answer kept on run.rows
[stub · run] recipe: (panel_1=/anthropic/claude-fable-5($question)!'Answer the question', panel_2=/openai/gpt-5.5($question)!'Answe…
Run · fable_plus_gpt · draco · 5 rows · answer_calls=15 

[stub · run] 1 rows × (2 panel + 1 synth) → 3 engine calls in a real run · every answer kept on run.rows
[stub · run] recipe: (panel_1=/anthropic/claude-fable-5($question)!'Answer the question', panel_2=/openai/gpt-5.5($question)!'Answe…
Run · fable_plus_gpt · healthbench-sample · 1 rows · answer_calls=3


## 4 · GRADE — the examiner marks the work *(real grader; judge LLM call stubbed)*

One `run.grade()` seam, two examiner strategies — and the difference between them is
the whole story of modern evals:

| | answer key | weighted rubric |
|---|---|---|
| examiner | `MultipleChoice` — letter vs key | `RubricJudge` — LLM marks each weighted criterion MET/UNMET |
| cost | free, instant | **real LLM calls** — chunked ≈ 6/row, official ≈ 53×runs/row |
| trust | deterministic | judge is **pinned**: model + temp 0.2 + reasoning low + no tools. Swap the judge and it's not "DRACO" anymore |

Grading is a pipeline stage with a bill, not post-processing: in the validated run the
**judge spend dominated — $2,699 vs $787 for answers**. That's why `run.cost` splits
answers vs judge (and billed vs cached, for the "$0 re-run" story).

The grading below runs the **real validated grader** — only the judge LLM call itself
is a deterministic stub. A grader that can't parse the judge's output returns an
explicit `unresolved` — never a made-up number.

In [5]:
# Answer-key path: pure code, zero judge calls
scores_mx = run_mx.grade()          # mcq rows default to MultipleChoice
scores_mx

[stub · grade] MultipleChoice — pure python, 0 judge calls, $0 · prod: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/mcq_exact.py


Scores · medxpertqa-sample · MultipleChoice · 1.000 ± 0.000 (n=2)

In [6]:
# Rubric path: the protocol pins are part of the result identity
judge = sf.RubricJudge(
    model="google/gemini-3.1-pro-preview",
    temperature=0.2,
    reasoning="low",
    mode="chunked",     # "official" for paper-faithful numbers (53×runs calls/row)
    runs=1,
)

scores_draco = run_draco.grade(judge)
print(repr(scores_draco), "\n")

# THE GENERICITY GATE — a second rubric benchmark, zero new grading code:
scores_hb = run_hb.grade(sf.RubricJudge(model="openai/gpt-5.4", mode="chunked", runs=1))
print(repr(scores_hb))

[stub · grade] REAL grader (benchmarking/graders/rubric.py — the one behind the $233.78 run of record) · 29 judge calls, stubbed deterministically; in prod each is one pinned call to google/gemini-3.1-pro-preview
[stub · grade] prod grader: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/rubric.py · judge prompts: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/prompts/rubric.py
Scores · draco · RubricJudge(chunked, runs=1) · 0.534 ± 0.136 (n=5) · pass_rate 0.64 · verdict_coverage 1.11 

[stub · grade] REAL grader (benchmarking/graders/rubric.py — the one behind the $233.78 run of record) · 2 judge calls, stubbed deterministically; in prod each is one pinned call to openai/gpt-5.4
[stub · grade] prod grader: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/rubric.py · judge prompts: https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/prompts/rubric.py
Scores · 

## 5 · AGGREGATE — the report card *(stub counts + real compare)*

Mean over rows + error bars (never over-read a 5-row slice) + the honest bill. Then
the punchline against the full-100-row scores of record: **the fusion beats its best
own member by ~12 points.**

**The run of record behind `validated_run/`** (so this notebook is self-contained):
$233.78 for 100 rows (~$2.34/row), official grading with `judge_runs: 3`, and
`reuse_panel_answers: true` — panel answers were cached from the solo runs, so only
the synthesizer and judge were paid again. That reuse is what made the validation
affordable, and it's the pattern the SDK's caching must keep.

**Honesty rules when quoting numbers:**
- The 0.6862 required server-side web tools. **No exact no-tools number exists
  anywhere** — a tools-off demo score can be *labeled*, never *compared* to the chart.
- Even our validated run is **not yet a certified reproduction** — publication is
  gated on an official-mode pass with `judge_runs ≥ 5` (ours used 3). Present every
  number as **"our independent run"**, never "DRACO reproduced".

In [7]:
print("the bill, per run (stub counts — a real run adds billed vs cached USD):")
for name, r in (("medxpert", run_mx), ("draco", run_draco), ("healthbench", run_hb)):
    print(f"  {name:<12} {r.cost}")

print("\nthe claim, against the full-100-row run of record:")
import pandas as pd
df = pd.read_csv("validated_run/by_model.csv")
print(df[df.model.isin(["fusion:fable_plus_gpt", "openai/gpt-5.5", "anthropic/claude-fable-5"])][
    ["model", "normalized_score"]
].sort_values("normalized_score", ascending=False).to_string(index=False))

# week-3: sf.leaderboard("draco").diff(run_draco) — publish & compare live

the bill, per run (stub counts — a real run adds billed vs cached USD):
  medxpert     {'answer_calls': 6, 'judge_calls': 0, 'usd': None, 'note': 'stub run — a real run reports billed vs cached USD here'}
  draco        {'answer_calls': 15, 'judge_calls': 29, 'usd': None, 'note': 'stub run — a real run reports billed vs cached USD here'}
  healthbench  {'answer_calls': 3, 'judge_calls': 2, 'usd': None, 'note': 'stub run — a real run reports billed vs cached USD here'}

the claim, against the full-100-row run of record:
                   model  normalized_score
   fusion:fable_plus_gpt            0.6862
          openai/gpt-5.5            0.5722
anthropic/claude-fable-5            0.5322


## 6 · Extend it yourself — a brand-new benchmark in ~25 lines

How does *anyone else* generalize this to their benchmark? The same way
[inspect_ai](https://github.com/UKGovernmentBEIS/inspect_ai) does it: every stage of
`load → run → grade → aggregate` resolves its strategy from a **named registry**, and
you plug in with a decorator (`@sf.loader`, `@sf.runner`, `@sf.grader`,
`@sf.aggregator`) — exactly like inspect_ai's `@task` / `@solver` / `@scorer`, with
`record_to_row` playing the role of their `record_to_sample` for foreign data schemas.

Below: a benchmark type this notebook has never seen — **summarization**, checked by a
**keyword-coverage grader** (a "soft metric", one of the *later* grading modes from the
taxonomy — working today because the seam accepts any strategy). One new loader + one
new runner + one new grader, and the same pipeline runs it end to end:

In [8]:
import json

# --- LOAD: my data, my schema, my loader --------------------------------------------
MY_RECORDS = [
    {"text": "Mitochondria convert nutrients into ATP, the cell's energy currency.",
     "must_mention": ["mitochondria", "energy"]},
    {"text": "The url4 engine fans a question out to a panel of models and reduces.",
     "must_mention": ["panel", "reduce"]},
]

@sf.loader("my-summaries")
def load_my_summaries(source, **kw):
    rows = [sf.Row(f"sum-{i}", "my-summaries", "summarize",
                   f"Summarize in one sentence: {r['text']}",
                   json.dumps(r["must_mention"]))          # answer = my own key format
            for i, r in enumerate(MY_RECORDS, 1)]
    return sf.Benchmark("my-summaries", "summarize", rows)

# --- RUN: a new task_type needs one runner strategy ----------------------------------
@sf.runner("summarize")
def run_summarize(fusion, row, seed):
    answer = f"(stub one-sentence summary of {row.id}: mitochondria panel reduce energy)"
    return {m: answer for m in fusion.models}, answer

# --- GRADE: a new checking mode = one grader with grade_row(row, answer) -> [0, 1] ----
@sf.grader("keyword-coverage")
class KeywordCoverage:
    def grade_row(self, row, answer):
        keywords = json.loads(row.answer)
        return sum(k.lower() in answer.lower() for k in keywords) / len(keywords)

# --- the SAME four steps, zero pipeline changes --------------------------------------
bench_mine = sf.benchmark("my-summaries")
run_mine = fusion_mcq.evaluate(bench_mine, seed=0)
scores_mine = run_mine.grade(KeywordCoverage())
print(repr(scores_mine), "\n")

sf.registry()   # the four seams, live — every strategy currently pluggable

[stub · run] 2 rows × (3 panel + 0 synth) → 6 engine calls in a real run · every answer kept on run.rows
[stub · run] recipe: (panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question', panel_2=/gemini/2.5($question)!'Answ…
[stub · grade] KeywordCoverage — pure python, 0 judge calls, $0
Scores · my-summaries · KeywordCoverage · 1.000 ± 0.000 (n=2) 

loader      .jsonl, healthbench-sample, medxpertqa-sample, my-summaries
runner      abstractive, mcq, summarize
grader      keyword-coverage, multiple-choice, rubric-judge
aggregator  mean


## Recap — why this generalizes

Three benchmarks that could not look more different — MCQ medicine, open-ended
research, patient conversations — ran through the **same four steps**, and the second
rubric benchmark needed **zero new grading code** (that's the genericity gate). Every
benchmark is one pick from each column, and each column is a pluggable seam:

![Every benchmark = one pick from each column](diagrams/sf-benchmark-taxonomy.png)

A new kind of benchmark — including ones that don't exist yet — is **one new strategy
at one seam**, never a pipeline reshape: a new way of working → a Runner; a new way of
checking → a Grader; a new way of rolling up → an Aggregator; a new data source → a
loader (§6 showed all three in ~25 lines).

### What was a stub, and where the real thing lives

All stubs are in [`sf_stubs.py`](sf_stubs.py) — one file to delete as the SDK catches up:

| Stub | SDK gap | Production reference (screamingface-benchmarks) |
|---|---|---|
| `sf.benchmark(...)` loader + Row/rubric preview | 1 | [`data_ingestion/run_ingestion.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/data_ingestion/run_ingestion.py) · [`data_ingestion/hf_loader.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/data_ingestion/hf_loader.py) |
| per-task-type prompts (MCQ vs research instructions) | 2 | [`benchmarking/prompts/`](https://github.com/OpenMined/screamingface-benchmarks/tree/main/benchmarking/prompts) — `mcq.py`, `open_ended.py`, `rubric.py`, `healthbench.py` |
| stub engine answers (+ `run.rows` kept) | 3 | url4 engine execution (real today for `fusion.evaluate("gpqa")`) |
| the judge LLM call inside `sf.RubricJudge` | 4 | graders are REAL: [`graders/rubric.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/rubric.py) (DRACO) · [`graders/healthbench_rubric.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/healthbench_rubric.py) (HealthBench) · [`graders/mcq_exact.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/benchmarking/graders/mcq_exact.py) (MedXpertQA) |
| `run.cost` call counts | 5 | [`runners/telemetry.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/runners/telemetry.py) + [`runners/spend_guard.py`](https://github.com/OpenMined/screamingface-benchmarks/blob/main/runners/spend_guard.py) (fail-closed budgets) |
| DRACO lineup preview recipe | 6 | model catalog + `url4.dev.toml` mock routes (this repo) |

Already real with no stub: `sf.Fusion` / `sf.MajorityVote` / `sf.Synthesize` /
`fusion.url4` compilation, the mock/live session, and the validated rubric grader in
this folder.

Questions → Khoa (or Siddhant) — DRACO/SOTA validation track.